# Graph Neural Networks with PyTorch Geometric
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/08_Graphs_Networks/gnn_intro_pytorch_geometric.ipynb)

GNNs learn from graph STRUCTURE: each layer lets every node aggregate messages from its neighbors, so embeddings encode both features and connectivity.

Task: classify papers by subject on Cora (2,708 nodes, 7 classes, citation edges) with a 2-layer GCN - the 'hello world' of GNNs. Runs on free Colab CPU in ~1 min.

In [ ]:
!pip install -q torch_geometric

## 1. Load Cora

In [ ]:
import torch
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures

dataset = Planetoid(root="data/Cora", name="Cora",
                    transform=NormalizeFeatures())
data = dataset[0]
print(dataset)
print("nodes/edges/classes/features:",
      data.num_nodes, data.num_edges, dataset.num_classes, data.num_node_features)
print("train mask:", int(data.train_mask.sum()), "nodes labeled for training")

Node features here are bag-of-words vectors; edges are citations. Standard transductive setup: train on masked nodes, evaluate on hidden ones.

## 2. Define the GCN

In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self, hidden=16):
        super().__init__()
        self.conv1 = GCNConv(dataset.num_features, hidden)
        self.conv2 = GCNConv(hidden, dataset.num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()      # message passing round 1
        x = F.dropout(x, training=self.training, p=0.5)
        return self.conv2(x, edge_index)          # round 2 -> class logits

model = GCN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
print(model)

## 3. Train + evaluate

In [ ]:
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward(); optimizer.step()
    return loss.item()

@torch.no_grad()
def acc(mask):
    model.eval()
    pred = model(data.x, data.edge_index).argmax(dim=1)
    return (pred[mask] == data.y[mask]).float().mean().item()

for epoch in range(1, 151):
    loss = train()
    if epoch % 30 == 0:
        print(f"epoch {epoch:3d}  loss={loss:.4f}  "
              f"train={acc(data.train_mask):.3f}  val={acc(data.val_mask):.3f}")
print(f"TEST accuracy: {acc(data.test_mask):.3f}")

Compare: logistic regression on raw features gets ~55% on Cora; the GCN's neighbor aggregation pushes past 80%.

## 4. See the learned embedding space

In [ ]:
@torch.no_grad()
def embed():
    model.eval()
    h = model.conv1(data.x, data.edge_index).relu()
    return h

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
z = PCA(n_components=2).fit_transform(embed().numpy())

plt.figure(figsize=(7, 5))
plt.scatter(z[:, 0], z[:, 1], c=data.y.numpy(), cmap="tab10", s=8)
plt.title("GCN embeddings - classes separate by structure+features")
plt.show()

## Where to go next
| Layer | Idea |
|---|---|
| `GATConv` | attention-weighted neighbors |
| `SAGEConv` | sampled neighborhoods -> billion-scale (GraphSAGE) |
| `global_mean_pool` | node -> GRAPH classification |
| link prediction | decode edge scores from node embeddings |

Rule of thumb: 2-3 layers max (over-smoothing), always keep a validation mask.